# FMQE - FileMaker to PostgreSQL Incremental Import

This notebook provides tools for importing Quantum data elements from FileMaker to PostgreSQL.

**Key Features:**
- First import brings ALL data elements including auditing attributes
- Subsequent imports use auditing attributes (devModificationTimestamp) to only import changed data
- Import history tracking for audit trail

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import pandas as pd
from datetime import datetime

# =============================================================================
# CRITICAL: JVM Setup - MUST run before any JDBC operations
# =============================================================================

# Set JAVA_HOME for JDBC driver
JAVA_HOME = r'C:\projects\duft\java\jdk-21.0.4+7'
os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = JAVA_HOME + r'\bin;' + os.environ.get('PATH', '')

# Set JVM path explicitly - this is the key fix for "jvm.dll not found"
JVM_PATH = os.path.join(JAVA_HOME, 'bin', 'server', 'jvm.dll')

# Verify JVM exists
if not os.path.exists(JVM_PATH):
    raise FileNotFoundError(f"JVM not found at: {JVM_PATH}")

# Get absolute path to JDBC driver JAR
JDBC_JAR = os.path.join(os.path.dirname(os.path.abspath(__file__ if '__file__' in dir() else '.')), 'fmjdbc.jar')
if not os.path.exists(JDBC_JAR):
    JDBC_JAR = os.path.abspath('fmjdbc.jar')
if not os.path.exists(JDBC_JAR):
    JDBC_JAR = r'C:\projects\duft\fmqe\fmjdbc.jar'

# Initialize JPype with the correct JVM path and JDBC driver in classpath
# This MUST happen before importing jaydebeapi
import jpype
import jpype.imports

if not jpype.isJVMStarted():
    print(f"Starting JVM from: {JVM_PATH}")
    jpype.startJVM(JVM_PATH, classpath=[JDBC_JAR], convertStrings=True)
    print("JVM started successfully!")
else:
    print("JVM already running")

# Now import jaydebeapi (after JVM is started)
import jaydebeapi
from sqlalchemy import create_engine, text

# Add current directory to path
sys.path.insert(0, os.getcwd())

print(f"\nWorking Directory: {os.getcwd()}")
print(f"JAVA_HOME: {JAVA_HOME}")
print(f"JVM Path: {JVM_PATH}")
print(f"JDBC JAR: {JDBC_JAR}")
print(f"JVM Started: {jpype.isJVMStarted()}")
print(f"Timestamp: {datetime.now()}")

## 2. Configuration

In [ ]:
# PostgreSQL Configuration
PG_CONFIG = {
    "server": "127.0.0.1",
    "username": "postgres",
    "password": "postgres",
    "port": "5432",
    "database": "qe_data"
}

# FileMaker Configuration
FM_CONFIG = {
    "server": "jdbc:filemaker://localhost/QuantumMaxuililiClinic",
    "username": "Administrator",
    "password": "Qepms!@dmin",
    "driver_jar": "fmjdbc.jar",
    "driver_class": "com.filemaker.jdbc.Driver"
}

# Schema names
STAGING_SCHEMA = "fm_staging"
PRODUCTION_SCHEMA = "fm_data"

print("Configuration loaded successfully")

## 3. PostgreSQL Connection

In [ ]:
def create_pg_database_if_not_exists():
    """Create the qe_data database if it doesn't exist."""
    default_url = f"postgresql://{PG_CONFIG['username']}:{PG_CONFIG['password']}@{PG_CONFIG['server']}:{PG_CONFIG['port']}/postgres"
    default_engine = create_engine(default_url, isolation_level='AUTOCOMMIT')
    
    with default_engine.connect() as conn:
        result = conn.execute(text(f"SELECT 1 FROM pg_database WHERE datname = '{PG_CONFIG['database']}'")).fetchone()
        if not result:
            conn.execute(text(f"CREATE DATABASE {PG_CONFIG['database']}"))
            print(f"Database '{PG_CONFIG['database']}' created successfully")
        else:
            print(f"Database '{PG_CONFIG['database']}' already exists")
    
    default_engine.dispose()

# Create database if needed
create_pg_database_if_not_exists()

In [ ]:
# Connect to PostgreSQL
pg_url = f"postgresql://{PG_CONFIG['username']}:{PG_CONFIG['password']}@{PG_CONFIG['server']}:{PG_CONFIG['port']}/{PG_CONFIG['database']}"
pg_engine = create_engine(pg_url)

# Test connection
with pg_engine.connect() as conn:
    result = conn.execute(text("SELECT version()")).fetchone()
    print(f"Connected to PostgreSQL: {result[0][:50]}...")

## 4. Setup Schemas and Import Tracking

In [ ]:
def setup_schemas():
    """Create necessary schemas."""
    with pg_engine.connect() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {STAGING_SCHEMA}"))
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {PRODUCTION_SCHEMA}"))
        conn.commit()
        print(f"Schemas created: {STAGING_SCHEMA}, {PRODUCTION_SCHEMA}")

def setup_import_tracking():
    """Create import history tracking table."""
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {STAGING_SCHEMA}.import_history (
        id SERIAL PRIMARY KEY,
        table_name VARCHAR(255) NOT NULL,
        import_type VARCHAR(50) NOT NULL,
        records_imported INTEGER NOT NULL,
        last_modification_timestamp TIMESTAMP,
        import_started_at TIMESTAMP NOT NULL,
        import_completed_at TIMESTAMP,
        status VARCHAR(50) DEFAULT 'running',
        error_message TEXT
    )
    """
    with pg_engine.connect() as conn:
        conn.execute(text(create_table_sql))
        conn.commit()
        print("Import tracking table created")

setup_schemas()
setup_import_tracking()

## 5. FileMaker Connection

In [ ]:
# FileMaker JDBC Driver path - uses JDBC_JAR from cell 2
driver_jar_path = JDBC_JAR  # Set in setup cell
driver_class = FM_CONFIG['driver_class']

# Check if driver exists
if os.path.exists(driver_jar_path):
    print(f"FileMaker JDBC driver found: {driver_jar_path}")
else:
    print(f"WARNING: FileMaker JDBC driver NOT found at: {driver_jar_path}")
    print("Please copy fmjdbc.jar to the current directory")

In [ ]:
def open_filemaker_connection():
    """Open connection to FileMaker database."""
    try:
        connection = jaydebeapi.connect(
            driver_class,
            FM_CONFIG['server'],
            [FM_CONFIG['username'], FM_CONFIG['password']],
            driver_jar_path
        )
        print(f"Connected to FileMaker: {FM_CONFIG['server']}")
        return connection
    except Exception as e:
        print(f"Error connecting to FileMaker: {e}")
        return None

def close_filemaker_connection(conn):
    """Close FileMaker connection."""
    if conn:
        conn.close()
        print("FileMaker connection closed")

def fetch_filemaker_data(query, conn):
    """Execute query and return results as DataFrame."""
    cursor = conn.cursor()
    try:
        cursor.execute(query)
        columns = [desc[0] for desc in cursor.description]
        data = cursor.fetchall()
        df = pd.DataFrame(data, columns=columns)
        print(f"Fetched {len(df)} rows")
        return df
    finally:
        cursor.close()

In [ ]:
# Connect to FileMaker
fm_connection = open_filemaker_connection()

## 6. Table Definitions for Import

Define the tables to import with their queries and auditing columns.

In [ ]:
# Table definitions with audit columns for incremental imports
# NOTE: Some tables use auditModTimestamp, others use devModificationTimestamp
TABLES = [
    {
        "name": "pat",
        "query": "SELECT id, artNumber, artNumberLegacy, auditCreateTimestamp, auditModTimestamp FROM pat",
        "primary_key": "id",
        "audit_column": "auditModTimestamp"
    },
    {
        "name": "cd",
        "query": "SELECT idPatient, hivConfirmationDate, hivConfirmatoryResultsDate, hivConfirmatoryResultsType, idFacilityCreate, fullDisclosureDate, hivEnrolledDate, disclosureEnrollmentDate, artEligibleReason, artStartDate, idFacilityARTStart, devCreationTimestamp, devModificationTimestamp FROM cd",
        "primary_key": "idPatient",
        "audit_column": "devModificationTimestamp"
    },
    {
        "name": "fup",
        "query": "SELECT id, idPatient, idFacilityCreate, visitDate, followUpDate, scheduledDate, careModel, pregnantStatus, Breastfeeding, pregnantLmp, pregnantEdd, ctxAdherence, arvAdherence, oiDetail, oiOther, tbScreenResult, auditCreateTimestamp, auditModTimestamp FROM fup",
        "primary_key": "id",
        "audit_column": "auditModTimestamp"
    },
    {
        "name": "tsfr",
        "query": "SELECT id, idPatient, status, \"date\", devCreationTimestamp, devModificationTimestamp FROM tsfr",
        "primary_key": "id",
        "audit_column": "devModificationTimestamp"
    },
    {
        "name": "meas",
        "query": "SELECT id, idPatient, CAST(weight AS VARCHAR) AS weight, whoStage, \"date\", devCreationTimestamp, devModificationTimestamp FROM meas",
        "primary_key": "id",
        "audit_column": "devModificationTimestamp"
    },
    {
        "name": "rgm",
        "query": "SELECT id, idPatient, idFacilityCreate, \"date\", type, line, code, dosage, duration, reason, auditCreateTimestamp, auditModTimestamp FROM rgm",
        "primary_key": "id",
        "audit_column": "auditModTimestamp"
    },
    {
        "name": "ti",
        "query": "SELECT id, idPatient, idFacilityCreate, interruptionDate, CAST(interruptionReason AS VARCHAR) AS interruptionReason, interruptionReasonOther, restartDate, duration, devCreationTimestamp, devModificationTimestamp FROM ti",
        "primary_key": "id",
        "audit_column": "devModificationTimestamp"
    },
    {
        "name": "tbt",
        "query": "SELECT id, idPatient, idFacilityCreate, category, startDate, regimen, duration, stopDateExpected, stopDateActual, registrationNumber, site, siteDetail, CASE WHEN outcome = '1' THEN 'Cured' WHEN outcome = '2' THEN 'Treatment complete' WHEN outcome = '3' THEN 'Died' WHEN outcome = '4' THEN 'Failure' WHEN outcome = '5' THEN 'Lost to follow-up' WHEN outcome = '6' THEN 'Not evaluated' ELSE outcome END AS outcome, devCreationTimestamp, devModificationTimestamp FROM tbt",
        "primary_key": "id",
        "audit_column": "devModificationTimestamp"
    },
    # NOTE: tpt table does not exist in this FileMaker database
    {
        "name": "lab",
        "query": "SELECT l.id, l.idPatient, l.idFacilityCreate, l.orderDate, l.result, CAST(l.resultDate AS VARCHAR) AS resultDate, lm.name AS test_name, l.status, l.auditCreateTimestamp, l.auditModTimestamp FROM lab l INNER JOIN LabMaster lm ON lm.id = l.idTest",
        "primary_key": "id",
        "audit_column": "auditModTimestamp"
    }
]

print(f"Configured {len(TABLES)} tables for import")
for t in TABLES:
    print(f"  - {t['name']} (PK: {t['primary_key']}, Audit: {t['audit_column']})")

## 7. Import Functions

In [ ]:
def get_last_import_timestamp(table_name):
    """Get the last successful import timestamp for a table."""
    query = f"""
    SELECT last_modification_timestamp
    FROM {STAGING_SCHEMA}.import_history
    WHERE table_name = :table_name
      AND status = 'completed'
    ORDER BY import_completed_at DESC
    LIMIT 1
    """
    with pg_engine.connect() as conn:
        result = conn.execute(text(query), {'table_name': table_name}).fetchone()
    
    if result and result[0]:
        return result[0]
    return None

def record_import(table_name, import_type, records, last_ts, status='completed', error=None):
    """Record an import in the history table."""
    insert_sql = f"""
    INSERT INTO {STAGING_SCHEMA}.import_history 
    (table_name, import_type, records_imported, last_modification_timestamp, 
     import_started_at, import_completed_at, status, error_message)
    VALUES (:table_name, :import_type, :records, :last_ts, 
            :started, :completed, :status, :error)
    """
    now = datetime.now()
    with pg_engine.connect() as conn:
        conn.execute(text(insert_sql), {
            'table_name': table_name,
            'import_type': import_type,
            'records': records,
            'last_ts': last_ts,
            'started': now,
            'completed': now,
            'status': status,
            'error': error
        })
        conn.commit()

In [ ]:
from datetime import timedelta

def import_table(table_config, fm_conn, force_full=False):
    """
    Import a single table from FileMaker to PostgreSQL.
    
    - First import: Brings ALL data including audit columns
    - Subsequent imports: Uses audit timestamp to get only changed records
    """
    table_name = table_config['name']
    base_query = table_config['query']
    audit_column = table_config.get('audit_column', 'devModificationTimestamp')
    primary_key = table_config.get('primary_key')
    
    print(f"\n{'='*60}")
    print(f"Importing table: {table_name}")
    print(f"{'='*60}")
    
    # Check for last import timestamp
    last_ts = None if force_full else get_last_import_timestamp(table_name)
    import_type = 'full' if last_ts is None else 'incremental'
    
    print(f"Import type: {import_type}")
    if last_ts:
        print(f"Last import timestamp: {last_ts}")
    
    try:
        # Build query
        if import_type == 'incremental' and audit_column:
            ts_str = last_ts.strftime('%m/%d/%Y %H:%M:%S')
            # Use > to only get records AFTER the last imported timestamp
            query = f"{base_query} WHERE {audit_column} > '{ts_str}'"
        else:
            query = base_query
        
        print(f"Executing query...")
        df = fetch_filemaker_data(query, fm_conn)
        
        if df.empty:
            print(f"No new records to import")
            record_import(table_name, import_type, 0, last_ts)
            return {'table': table_name, 'records': 0, 'type': import_type, 'status': 'completed'}
        
        # Get max modification timestamp and add 1 second buffer
        # This prevents re-importing records at exactly the cutoff time
        new_last_ts = None
        if audit_column and audit_column in df.columns:
            max_ts = df[audit_column].max()
            if pd.notna(max_ts):
                max_ts_dt = pd.to_datetime(max_ts) if isinstance(max_ts, str) else max_ts
                # Add 1 second so next import doesn't re-fetch records at this exact timestamp
                new_last_ts = max_ts_dt + timedelta(seconds=1)
                print(f"Max timestamp in data: {max_ts_dt}")
                print(f"Stored cutoff (max + 1s): {new_last_ts}")
        
        # Write to PostgreSQL
        staging_table = f"{STAGING_SCHEMA}.{table_name}"
        
        if import_type == 'full':
            # Full import: replace entire table
            df.to_sql(table_name, pg_engine, schema=STAGING_SCHEMA, 
                      if_exists='replace', index=False)
        else:
            # Incremental: upsert if PK exists, else append
            if primary_key and primary_key in df.columns:
                with pg_engine.connect() as conn:
                    # Filter out None/NULL primary key values
                    pk_values = [v for v in df[primary_key].tolist() if v is not None and pd.notna(v)]
                    
                    if pk_values:
                        # Build safe placeholders - only include non-null values
                        placeholders = ', '.join([f"'{v}'" if isinstance(v, str) else str(v) for v in pk_values])
                        delete_sql = f"DELETE FROM {staging_table} WHERE {primary_key} IN ({placeholders})"
                        print(f"Deleting {len(pk_values)} existing records for upsert...")
                        conn.execute(text(delete_sql))
                        conn.commit()
                    
                    # Warn about records with NULL primary keys
                    null_pk_count = len(df) - len(pk_values)
                    if null_pk_count > 0:
                        print(f"WARNING: {null_pk_count} records have NULL primary key - these will be appended without deduplication")
            
            df.to_sql(table_name, pg_engine, schema=STAGING_SCHEMA, 
                      if_exists='append', index=False)
        
        records = len(df)
        print(f"Successfully imported {records} records to {staging_table}")
        
        record_import(table_name, import_type, records, new_last_ts)
        
        return {'table': table_name, 'records': records, 'type': import_type, 'status': 'completed'}
        
    except Exception as e:
        error_msg = str(e)
        print(f"ERROR: {error_msg}")
        record_import(table_name, import_type, 0, last_ts, status='failed', error=error_msg)
        return {'table': table_name, 'records': 0, 'type': import_type, 'status': 'failed', 'error': error_msg}

## 8. Run Import (Auto-Detects Full vs Incremental)

This cell automatically detects whether to run a full or incremental import:
- **First run (no history)**: Full import of all data
- **Subsequent runs**: Incremental import of only changed records

In [ ]:
# Run SMART IMPORT - Auto-detects full vs incremental
print("Starting IMPORT (auto-detect mode)...")
print(f"Timestamp: {datetime.now()}")
print()

results = []
for table_config in TABLES:
    # force_full=False lets the import_table function auto-detect
    # It will do FULL import if no history exists, INCREMENTAL if history exists
    result = import_table(table_config, fm_connection, force_full=False)
    results.append(result)

# Summary
print("\n" + "="*60)
print("IMPORT SUMMARY")
print("="*60)
total_records = 0
full_count = 0
incremental_count = 0
for r in results:
    status = "✓" if r['status'] == 'completed' else "✗"
    print(f"{status} {r['table']}: {r['records']} records ({r['type']})")
    total_records += r['records']
    if r['type'] == 'full':
        full_count += 1
    else:
        incremental_count += 1

print(f"\nTotal records imported: {total_records}")
print(f"Tables with full import: {full_count}")
print(f"Tables with incremental import: {incremental_count}")

## 9. Force Full Import (Optional)

Only run this cell if you need to force a complete re-import of all data, ignoring the import history.

In [ ]:
# OPTIONAL: Force FULL import - Only run this cell manually if needed!
# This will re-import ALL data regardless of import history

# Uncomment the lines below to force a full import:
# print("Starting FORCED FULL IMPORT...")
# print(f"Timestamp: {datetime.now()}")
# results = []
# for table_config in TABLES:
#     result = import_table(table_config, fm_connection, force_full=True)
#     results.append(result)
# print(f"Full import completed: {sum(r['records'] for r in results)} total records")

print("⚠️  This cell is commented out by default.")
print("To force a full import, uncomment the code above and run this cell manually.")

## 10. View Import History

In [ ]:
# View import history - latest import per table
latest_query = f"""
SELECT DISTINCT ON (table_name) 
    table_name, import_type, records_imported, 
    last_modification_timestamp, import_completed_at, status
FROM {STAGING_SCHEMA}.import_history
WHERE status = 'completed'
ORDER BY table_name, import_completed_at DESC
"""

print("=== Latest Import Per Table ===")
latest_df = pd.read_sql(latest_query, pg_engine)
display(latest_df)

# Full history (last 20 entries)
print("\n=== Recent Import History (All Runs) ===")
history_query = f"""
SELECT table_name, import_type, records_imported, 
       last_modification_timestamp, import_completed_at, status
FROM {STAGING_SCHEMA}.import_history
ORDER BY import_completed_at DESC
LIMIT 20
"""
history_df = pd.read_sql(history_query, pg_engine)
history_df

## 11. View Imported Data

In [ ]:
# Check record counts in staging schema
print("Record counts in staging schema:")
print("-" * 40)

for table_config in TABLES:
    table_name = table_config['name']
    try:
        count_query = f"SELECT COUNT(*) FROM {STAGING_SCHEMA}.{table_name}"
        with pg_engine.connect() as conn:
            count = conn.execute(text(count_query)).fetchone()[0]
        print(f"{table_name}: {count:,} records")
    except Exception as e:
        print(f"{table_name}: Not found or error")

In [ ]:
# DEBUG: Compare specific record between FileMaker and PostgreSQL
LEGACY_ART_NUMBER = "597051700009"

print(f"=== Checking patient with artNumberLegacy: {LEGACY_ART_NUMBER} ===\n")

# Query FileMaker
print("--- FileMaker Record ---")
fm_query = f"SELECT id, artNumber, artNumberLegacy, auditCreateTimestamp, auditModTimestamp FROM pat WHERE artNumberLegacy = '{LEGACY_ART_NUMBER}'"
try:
    fm_df = fetch_filemaker_data(fm_query, fm_connection)
    if fm_df.empty:
        print("NOT FOUND in FileMaker")
    else:
        for col in fm_df.columns:
            print(f"{col}: {fm_df[col].iloc[0]}")
except Exception as e:
    print(f"Error querying FileMaker: {e}")

# Query PostgreSQL
print("\n--- PostgreSQL Record ---")
pg_query = f"SELECT id, \"artNumber\", \"artNumberLegacy\", \"auditCreateTimestamp\", \"auditModTimestamp\" FROM fm_staging.pat WHERE \"artNumberLegacy\" = '{LEGACY_ART_NUMBER}'"
try:
    pg_df = pd.read_sql(pg_query, pg_engine)
    if pg_df.empty:
        print("NOT FOUND in PostgreSQL")
    else:
        for col in pg_df.columns:
            print(f"{col}: {pg_df[col].iloc[0]}")
except Exception as e:
    print(f"Error querying PostgreSQL: {e}")

# Compare timestamps
print("\n--- Comparison ---")
print(f"Last import cutoff: {get_last_import_timestamp('pat')}")
if not fm_df.empty and 'auditModTimestamp' in fm_df.columns:
    fm_ts = fm_df['auditModTimestamp'].iloc[0]
    last_import = get_last_import_timestamp('pat')
    print(f"FileMaker auditModTimestamp: {fm_ts}")
    if fm_ts and last_import:
        fm_ts_dt = pd.to_datetime(fm_ts) if isinstance(fm_ts, str) else fm_ts
        if fm_ts_dt > last_import:
            print("✓ FileMaker timestamp IS NEWER - should be picked up")
        else:
            print("✗ FileMaker timestamp IS OLDER - will NOT be picked up")
            print("  -> The record modification did NOT update auditModTimestamp in FileMaker!")

## 12. Cleanup

In [ ]:
# Close FileMaker connection when done
close_filemaker_connection(fm_connection)

# Dispose PostgreSQL engine
pg_engine.dispose()

print("All connections closed")